In [17]:
# uploading file

import ipywidgets as widgets
from IPython.display import display

# Create upload box
upload_box = widgets.FileUpload(

    # Allow only Excel files
    accept=".xlsx,.xls",

    # upload only one file
    multiple=False
)

# Display upload box on screen
display(upload_box)

FileUpload(value=(), accept='.xlsx,.xls', description='Upload')

In [18]:
# loading file into dataframe

import pandas as pd
import io

# Check if file is uploaded
if len(upload_box.value) == 0:
    print("No file uploaded.")

else:

    # Get uploaded file
    uploaded_file = upload_box.value[0]

    # Read Excel file
    df = pd.read_excel(
        io.BytesIO(uploaded_file["content"])
    )

    print("File Loaded Successfully")

File Loaded Successfully


In [19]:
# Clean column names
df.columns = df.columns.str.strip().str.lower()

# Keep required columns
df = df[['vehicle reg no', 'city', 'sub vendor rate']]

# Display available vehicles
vehicles = df['vehicle reg no'].unique()
print("\nAvailable Vehicles")
print("-" * 30)

for index, vehicle in enumerate(vehicles):
    print(f"{index} : {vehicle}")

# Select vehicle
vehicle_index = int(input("\nEnter Vehicle Index: "))
selected_vehicle = vehicles[vehicle_index]

print(f"\nSelected Vehicle: {selected_vehicle}")


Available Vehicles
------------------------------
0 : UP14PT4981
1 : HR55AV4586
2 : HR55AQ3285
3 : HR38AF4597



Enter Vehicle Index:  0



Selected Vehicle: UP14PT4981


In [20]:
# Filter vehicle data
vehicle_data = df[df['vehicle reg no'] == selected_vehicle]

# Create trip summary
trip_summary = (
    vehicle_data
    .groupby(['city', 'sub vendor rate'])
    .size()
    .reset_index(name='trip_count')
)

# Calculate amount
trip_summary['amount'] = (trip_summary['sub vendor rate']* trip_summary['trip_count'])

# Create total row
total_trips = trip_summary['trip_count'].sum()
total_amount = trip_summary['amount'].sum()

# Create total row
total_row = pd.DataFrame({
    'city': ['TOTAL'],
    'sub vendor rate': [''],
    'trip_count': [total_trips],
    'amount': [total_amount]
})

# Add total row 
trip_summary = pd.concat(
    [trip_summary, total_row],
    ignore_index=True
)

print("\nTrip Summary")
display(trip_summary)


Trip Summary


,city,sub vendor rate,trip_count,amount
0,Faridabad,1000,4,4000
1,Ghaziabad,1100,12,13200
2,Greater Noida,1200,2,2400
3,Gurugram,550,19,10450
4,Gurugram,750,10,7500
5,Gurugram,800,1,800
6,New Delhi,750,12,9000
7,New Delhi,800,8,6400
8,New Delhi,950,23,21850
9,Noida,1025,8,8200


In [21]:
# adding and deducting charges
penalty = float(input("Penalty: "))
office_charge = float(input("Office Charge: "))
mcd = float(input("MCD: "))
gps = float(input("GPS: "))
tds = float(input("TDS: "))
toll_tax = float(input("Toll Tax: "))

final_payable_amount = (
    total_amount
    + mcd
    + toll_tax
    - penalty
    - office_charge
    - gps
    - tds
)
# creating summary
payment_summary = pd.DataFrame({
    'Particular': [
        'Total Amount',
        'MCD',
        'Toll Tax',
        'Penalty',
        'Office Charge',
        'GPS',
        'TDS',
        'Final Payable Amount'
    ],
    'Amount': [
        total_amount,
        mcd,
        toll_tax,
        penalty,
        office_charge,
        gps,
        tds,
        final_payable_amount
    ]
})

print("\nPayment Summary")
display(payment_summary)

Penalty:  1
Office Charge:  2
MCD:  3
GPS:  4
TDS:  5
Toll Tax:  6



Payment Summary


,Particular,Amount
0,Total Amount,83800.0
1,MCD,3.0
2,Toll Tax,6.0
3,Penalty,1.0
4,Office Charge,2.0
5,GPS,4.0
6,TDS,5.0
7,Final Payable Amount,83797.0


In [22]:
# Generating pdf
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle
)
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER

month = input("Enter Month (e.g. June 2026): ").strip()


# PDF FILE
pdf_file = f"{selected_vehicle}_{month}_Bill.pdf"

doc = SimpleDocTemplate(
    pdf_file,
    rightMargin=30,
    leftMargin=30,
    topMargin=30,
    bottomMargin=30
)

styles = getSampleStyleSheet()

# CUSTOM STYLES
title_style = ParagraphStyle(
    "TitleStyle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=22,
    spaceAfter=20
)

info_style = ParagraphStyle(
    "InfoStyle",
    parent=styles["Normal"],
    alignment=TA_CENTER,
    fontSize=14,
    leading=20
)

elements = []

# REPORT TITLE
elements.append(
    Paragraph("VEHICLE BILLING REPORT", title_style)
)

# VEHICLE INFO BOX
info_data = [[
    f"Vehicle No: {selected_vehicle}",
    f"Month: {month}"
]]

info_table = Table(info_data, colWidths=[250, 250])

info_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, -1), colors.darkblue),
    ('TEXTCOLOR', (0, 0), (-1, -1), colors.white),
    ('FONTNAME', (0, 0), (-1, -1), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, -1), 14),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 12),
    ('TOPPADDING', (0, 0), (-1, -1), 12),
]))

elements.append(info_table)
elements.append(Spacer(1, 20))

# TRIP SUMMARY TITLE
elements.append(
    Paragraph("<b>Trip Summary</b>", styles["Heading2"])
)

trip_data = [trip_summary.columns.tolist()] + trip_summary.values.tolist()

trip_table = Table(
    trip_data,
    colWidths=[130, 130, 130, 130]
)

trip_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.darkblue),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 11),

    ('GRID', (0, 0), (-1, -1), 1, colors.black),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),

    ('ROWBACKGROUNDS',
     (0, 1), (-1, -1),
     [colors.whitesmoke, colors.lightgrey]),
]))

elements.append(trip_table)
elements.append(Spacer(1, 25))

# PAYMENT SUMMARY TITLE
elements.append(
    Paragraph("<b>Payment Summary</b>", styles["Heading2"])
)

payment_data = (
    [payment_summary.columns.tolist()]
    + payment_summary.values.tolist()
)

payment_table = Table(
    payment_data,
    colWidths=[260, 260]
)

payment_table.setStyle(TableStyle([
    ('BACKGROUND', (0, 0), (-1, 0), colors.darkgreen),
    ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
    ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
    ('FONTSIZE', (0, 0), (-1, 0), 11),

    ('GRID', (0, 0), (-1, -1), 1, colors.black),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),

    ('ROWBACKGROUNDS',
     (0, 1), (-1, -1),
     [colors.beige, colors.whitesmoke]),
]))

elements.append(payment_table)

elements.append(Spacer(1, 40))

# SIGNATURE
signature_table = Table(
    [["Prepared By", "Authorized Signature"]],
    colWidths=[260, 260]
)

signature_table.setStyle(TableStyle([
    ('LINEABOVE', (0, 0), (0, 0), 1, colors.black),
    ('LINEABOVE', (1, 0), (1, 0), 1, colors.black),
    ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
    ('TOPPADDING', (0, 0), (-1, -1), 25),
]))

elements.append(signature_table)

# GENERATE PDF
doc.build(elements)

print(f"PDF Generated Successfully!")
print(f"Saved As: {pdf_file}")

Enter Month (e.g. June 2026):  appp


PDF Generated Successfully!
Saved As: UP14PT4981_appp_Bill.pdf
